# Layer 7 — Text to Speech

This checkpoint uses Piper on the CPU to turn one sentence into a local WAV file. It does not call Groq. For continuous narration, `src.session` loads the voice once and reuses it across segments to avoid a model-loading pause after each group of sentences.

If the voice is missing, download it once from the project root:

```powershell
uv run python -m piper.download_voices --download-dir cache/voices en_US-lessac-medium
```

In [3]:
from pathlib import Path
from time import perf_counter
import sys
import wave

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Run this notebook from the ConversIQ project or notebooks directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Audio, display
from src.tts import synthesize_speech

voice_path = PROJECT_ROOT / 'cache' / 'voices' / 'en_US-lessac-medium.onnx'
output_path = PROJECT_ROOT / 'cache' / 'tts' / 'layer7_demo.wav'
demo_text = 'ConversIQ is ready to read your document aloud. You can ask a question at any time.'

if not voice_path.is_file():
    raise FileNotFoundError(f'Download the Piper voice model first: {voice_path}')
print(f'Voice model: {voice_path}')
print(f'Output WAV: {output_path}')

Voice model: D:\conversIQ_v3\cache\voices\en_US-lessac-medium.onnx
Output WAV: D:\conversIQ_v3\cache\tts\layer7_demo.wav


## Synthesize and inspect the WAV

Listen to the embedded audio and confirm the sentence is intelligible. The WAV metadata check confirms that Piper produced non-empty audio.

In [4]:
started = perf_counter()
audio_path = synthesize_speech(demo_text, voice_path, output_path)
elapsed = perf_counter() - started

with wave.open(str(audio_path), 'rb') as wav_file:
    duration = wav_file.getnframes() / wav_file.getframerate()
    sample_rate = wav_file.getframerate()
    channels = wav_file.getnchannels()

assert duration > 0
print(f'Text: {demo_text}')
print(f'Generated file: {audio_path}')
print(f'Synthesis latency: {elapsed:.2f} s')
print(f'Audio duration: {duration:.2f} s; sample rate: {sample_rate} Hz; channels: {channels}')
display(Audio(filename=str(audio_path)))

Text: ConversIQ is ready to read your document aloud. You can ask a question at any time.
Generated file: D:\conversIQ_v3\cache\tts\layer7_demo.wav
Synthesis latency: 1.70 s
Audio duration: 4.92 s; sample rate: 22050 Hz; channels: 1
